# CPSA 06 — HaGRID teacher labels on the Kaggle GPU

Labels the HaGRID 30k-384p sample with the **RTMW-x whole-body teacher** from
`cpsa_ml.pseudolabel`, running ONNX Runtime on the T4 through
`CUDAExecutionProvider`.  The result is a `halfbust62` YOLO-pose dataset of
people facing the camera with large hands, many of them near the face, at the
deploy resolution.

**Inputs:** none attached; Internet on, GPU T4.  The notebook downloads the
sample from Hugging Face and the teacher weights from the rtmlib release.

**Runtime:** about 1.5-2 h on one T4: ~15 min for the 2 GB download and its
extraction, and roughly 45-90 min for detection and pose on the ~30k frames.

**Output:** `/kaggle/working/hagrid62.tar` (uncompressed; `data.yaml`,
`images/{train,val}` and `labels/{train,val}` at the archive root, 10 % of the
frames held out for validation) plus 16 rendered samples in
`/kaggle/working/samples/`.  Re-publish the tar as the Kaggle dataset
`fabiocabeccia/cpsa-hagrid62`.

**How notebook 04 attaches it:** attach `fabiocabeccia/cpsa-hagrid62` as a
dataset source, then run notebook 04.  It finds the attached `hagrid62.tar`,
extracts it with `cpsa_ml.datasets.staging.stage_archive` into
`/tmp/cpsa-hagrid62` and mixes that root into the training list through
`EXTRA_DATA` (`halfbust62` schema).

**Licenses:** HaGRID is distributed under a CC BY-SA 4.0 variant (attribution
and share-alike); keep that notice with the derived labels.  RTMW (the teacher
checkpoint) and rtmlib are Apache-2.0; `cpsa_ml` itself is MIT.  The labels are
machine-generated, so errors of the teacher propagate into the dataset.


In [ ]:
import subprocess
import sys
from importlib import metadata

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "0db3ffa7c4ec45de312fe9dd75ca10d94d42fdaf"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"cpsa-ml[pseudolabel] @ git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

# The pseudolabel extra pulls the CPU onnxruntime; the CUDA build must replace it.
try:
    metadata.version("onnxruntime")
except metadata.PackageNotFoundError:
    pass
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=True
    )

# rtmlib 0.0.16 is the release whose ONNX Runtime backend this notebook uses;
# onnxruntime-gpu[cuda,cudnn] brings the CUDA 12 runtime and cuDNN wheels.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "onnxruntime-gpu[cuda,cudnn]",
        "rtmlib==0.0.16",
    ],
    check=True,
)


In [ ]:
import onnxruntime as ort

# onnxruntime-gpu >= 1.19 loads the CUDA/cuDNN wheels installed above; older
# builds expose no preload_dlls and rely on the system libraries instead.
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()

PROVIDERS = ort.get_available_providers()
print("onnxruntime", ort.__version__)
print("providers:", PROVIDERS)

try:
    assert "CUDAExecutionProvider" in PROVIDERS, (
        f"CUDAExecutionProvider is missing from {PROVIDERS}"
    )

    # A tiny identity graph proves a real CUDA session works; the provider list
    # alone only says that the shared library was found.
    from onnx import TensorProto, helper

    graph = helper.make_graph(
        [helper.make_node("Identity", ["images"], ["poses"])],
        "cuda_preflight",
        [helper.make_tensor_value_info("images", TensorProto.FLOAT, [1, 1])],
        [helper.make_tensor_value_info("poses", TensorProto.FLOAT, [1, 1])],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    # onnx 1.23 writes IR 14 by default, which recent onnxruntime builds reject;
    # IR 10 is the version every ORT >= 1.15 accepts.
    model.ir_version = 10
    session = ort.InferenceSession(model.SerializeToString(), providers=["CUDAExecutionProvider"])
    session_providers = session.get_providers()
    print("preflight session:", session_providers)
    assert session_providers[0].startswith("CUDA"), session_providers

except Exception as exc:  # noqa: BLE001 - any failure must name the fallback
    raise RuntimeError(
        "ONNX Runtime CUDA is not usable in this kernel, so the teacher cannot run "
        "on the T4. Fallback (D3): run the same labeler locally on the CPU with "
        "`python -m cpsa_ml.pseudolabel --device cpu --teacher rtmw-l`. "
        f"Original error: {exc}"
    ) from exc


In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

DATA = Path("/tmp/hagrid")  # outside /kaggle/working: not part of the output
DATA.mkdir(parents=True, exist_ok=True)
ARCHIVE = DATA / "hagrid-sample-30k-384p.zip"
URL = (
    "https://huggingface.co/datasets/cj-mills/hagrid-sample-30k-384p"
    "/resolve/main/hagrid-sample-30k-384p.zip"
)
HAGRID_ROOT = DATA / "hagrid-sample-30k-384p" / "hagrid_30k"

if not ARCHIVE.is_file():
    # ~2 GB; curl -C - resumes the download when the session dies.
    subprocess.run(["curl", "-sSL", "--retry", "3", "-C", "-", "-o", str(ARCHIVE), URL], check=True)
if not HAGRID_ROOT.is_dir():
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall(DATA)

# os.walk, not rglob: the 30k tree has 30k directory entries and rglob stats
# every one of them twice.
image_count = sum(
    len([name for name in files if name.lower().endswith((".jpg", ".jpeg", ".png"))])
    for _, _, files in os.walk(HAGRID_ROOT)
)
print("HaGRID root:", HAGRID_ROOT)
print("HaGRID images:", image_count)
assert image_count >= 25000, f"expected the 30k sample, found {image_count} images"


In [ ]:
import subprocess
import sys
from pathlib import Path

OUT = Path("/tmp/hagrid_pl")  # resumable: a rerun skips the frames already labeled
# The labeler preloads the CUDA/cuDNN wheels in its own process and exits if
# ONNX Runtime falls back to the CPU, so this subprocess cannot run silently on
# the CPU; the streamed chunk log reports frames/s and an ETA.
# rtmlib 0.0.16 maps device "cuda" directly to "CUDAExecutionProvider";
# "cuda:0" instead builds a (provider, {"device_id": 0}) tuple. Both work, but
# the plain string is the value in rtmlib's RTMLIB_SETTINGS for the
# onnxruntime backend, so it is the one exercised here.
DEVICE = "cuda"
COMMAND = [
    sys.executable,
    "-m",
    "cpsa_ml.pseudolabel",
    "--frames",
    str(HAGRID_ROOT),
    "--out",
    str(OUT),
    "--device",
    DEVICE,
    "--teacher",
    "rtmw-x",
    "--no-hands",  # notebook 07 builds the hand crops; only halfbust62 is needed here
    "--val-fraction",
    "0.1",
]

print(" ".join(COMMAND), flush=True)
process = subprocess.Popen(
    COMMAND, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="", flush=True)  # streamed, so the Kaggle log shows progress
code = process.wait()
if code != 0:
    raise RuntimeError(f"auto-labeler exited with {code}: {' '.join(COMMAND)}")


In [ ]:
import json
from pathlib import Path

STATS = Path("/tmp/hagrid_pl/stats.json")
stats = json.loads(STATS.read_text(encoding="utf-8"))
print(json.dumps(stats, indent=2, sort_keys=True))

frames_per_split = {
    split: stats["splits"].get(split, {}).get("frames", 0) for split in ("train", "val")
}
labeled = sum(frames_per_split.values())
print("frames with a person:", frames_per_split, "total", labeled)
assert frames_per_split["train"] > 0, "the train split is empty"
assert labeled >= 25000, f"only {labeled} frames with a person, expected at least 25000"


In [ ]:
import random
from pathlib import Path

from PIL import Image, ImageDraw

from cpsa_ml.schema import HALFBUST_62

HALFBUST = Path("/tmp/hagrid_pl/halfbust")
SAMPLES = Path("/kaggle/working/samples")
SAMPLES.mkdir(parents=True, exist_ok=True)
for stale in SAMPLES.glob("*.png"):
    stale.unlink()

frames = [
    (image, HALFBUST / "labels" / split / f"{image.stem}.txt")
    for split in ("train", "val")
    for image in (HALFBUST / "images" / split).glob("*.jpg")
]
frames = [(image, label) for image, label in frames if label.is_file()]
assert len(frames) >= 16, f"only {len(frames)} labeled frames to render"
random.Random(0).shuffle(frames)

for index, (image_path, label_path) in enumerate(frames[:16]):
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for line in label_path.read_text(encoding="utf-8").splitlines():
        values = [float(value) for value in line.split()]
        if len(values) != 5 + 3 * len(HALFBUST_62.names):
            raise ValueError(f"{label_path}: unexpected label width {len(values)}")
        centre_x, centre_y, box_w, box_h = values[1:5]
        draw.rectangle(
            (
                (centre_x - box_w / 2) * width,
                (centre_y - box_h / 2) * height,
                (centre_x + box_w / 2) * width,
                (centre_y + box_h / 2) * height,
            ),
            outline=(255, 255, 255),
        )
        keypoints = values[5:]
        for point, name in enumerate(HALFBUST_62.names):
            x, y, visible = keypoints[3 * point : 3 * point + 3]
            if visible <= 0:
                continue
            colour = (0, 255, 0)  # body
            if "hand" in name:
                colour = (255, 128, 0)
            elif name.startswith("face"):
                colour = (0, 200, 255)
            draw.ellipse(
                (x * width - 3, y * height - 3, x * width + 3, y * height + 3), fill=colour
            )
    destination = SAMPLES / f"{index:02d}_{image_path.stem}.png"
    image.save(destination)
    print(destination.name, image.size)

written = sorted(SAMPLES.glob("*.png"))
print(f"{len(written)} samples in {SAMPLES}")
assert len(written) == 16, written


In [ ]:
import tarfile
from pathlib import Path

DATASET = Path("/tmp/hagrid_pl/halfbust")
TARBALL = Path("/kaggle/working/hagrid62.tar")

# Uncompressed on purpose: the Kaggle dataset mount reads it once, and an
# uncompressed tar lets stage_archive extract without a decompression pass.
with tarfile.open(TARBALL, "w") as archive:
    for path in sorted(DATASET.rglob("*")):
        archive.add(path, arcname=path.relative_to(DATASET).as_posix(), recursive=False)

with tarfile.open(TARBALL) as archive:
    members = archive.getmembers()
names = [member.name for member in members]
required = {"data.yaml", "images/train", "images/val", "labels/train", "labels/val"}
missing = sorted(required - set(names))
assert not missing, f"archive is missing {missing}"
assert (DATASET / "data.yaml").is_file()

print("archive:", TARBALL, round(TARBALL.stat().st_size / 2**20, 1), "MiB")
print("files:", sum(1 for member in members if member.isfile()))
for split in ("train", "val"):
    print(
        split,
        "images:",
        sum(1 for name in names if name.startswith(f"images/{split}/") and name.endswith(".jpg")),
        "labels:",
        sum(1 for name in names if name.startswith(f"labels/{split}/") and name.endswith(".txt")),
    )


## Next steps

1. Re-publish `/kaggle/working/hagrid62.tar` as the Kaggle dataset
   `fabiocabeccia/cpsa-hagrid62`.  Keep the derived labels under the same
   CC BY-SA 4.0 variant notice as HaGRID.
2. Attach that dataset to **notebook 04** (`cpsa-variant-study`); its staging
   cell extracts the tar with `cpsa_ml.datasets.staging.stage_archive` and
   passes the root in `EXTRA_DATA`, so the variant runs train on the HaGRID
   frames next to the COCO half-bust crops.
3. If the CUDA preflight above failed, use the D3 fallback on the local PC:
   `python -m cpsa_ml.pseudolabel --device cpu --teacher rtmw-l --frames
   <HaGRID root> --out /tmp/hagrid_pl --no-hands --val-fraction 0.1`, then
   publish the same `halfbust/data.yaml` tree as `fabiocabeccia/cpsa-hagrid62`.
